<a href="https://colab.research.google.com/github/lejuin/aidl_fingerspelling/blob/main/notebooks/Run_git_code_in_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import git repo

In [1]:
#!git clone https://github.com/lejuin/fingerspelling_asl.git
!git clone -b from_bucket https://github.com/lejuin/fingerspelling_asl.git

Cloning into 'fingerspelling_asl'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 73 (delta 18), reused 72 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 476.33 KiB | 2.57 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [2]:
!pip install -r fingerspelling_asl/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 130.3 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [3]:
import sys
sys.path.insert(0,'fingerspelling_asl')

# Setup for Colab

In [4]:
!curl ipinfo.io

{
  "ip": "34.125.153.255",
  "hostname": "255.153.125.34.bc.googleusercontent.com",
  "city": "Las Vegas",
  "region": "Nevada",
  "country": "US",
  "loc": "36.1750,-115.1372",
  "org": "AS396982 Google LLC",
  "postal": "89111",
  "timezone": "America/Los_Angeles",
  "readme": "https://ipinfo.io/missingauth"
}

In [5]:
from google.colab import auth
PROJECT_ID = "firststepsgc"
#PROJECT_ID = "buoyant-purpose-479417-t8"
auth.authenticate_user(project_id=PROJECT_ID)

In [7]:
# manually setting params
BUCKET_NAME = "aidl_asl_datasets"

args_dict = {
  'data_dir': f"gs://{BUCKET_NAME}/asl-fingerspelling",
  'train_csv': "train.csv",
  'run_name': None,
  'logdir':"artifacts/logs",
  'max_frames':160,
  'epochs':5,
  'batch_size':4,
  'lr':1e-3,
  'hidden_dim':128,
  'val_ratio':0.2,
  'seed':42,
  'train_size':200,
  'val_size':200,
  'max_phrase_len':0,
  'overfit_subset':0,
  'use_wandb':True,
  'wandb_project':"fingerspelling_asl",
  'wandb_entity':None,
  'wandb_run_name':None,
  'wandb_mode':"online",
  'wandb_tags':""
}

class Args:
    def __init__(self, dict_args):
        for key, value in dict_args.items():
            setattr(self, key, value)

args = Args(args_dict)

args.train_csv

'train.csv'

In [8]:
from src.train import *

In [9]:
import os
import re
import json
import argparse
from fsspec.implementations.local import LocalFileSystem
from datetime import datetime
from typing import List, Tuple

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

try:
    import wandb
except ImportError:
    wandb = None

from src.models.embedded_rnn import EmbeddedRNN
from src.data.dataset import ASLRightHandDataset, collate_fn
from src.utils.metrics import evaluate_metrics, ctc_greedy_decode
from src.utils.filesystem import get_filesystem, join_path

# Reproduce main function

In [10]:
fs = get_filesystem(args.data_dir)
train_csv = args.train_csv

if not os.path.isabs(train_csv):
    train_csv = join_path(fs, args.data_dir, train_csv)
vocab_json =join_path(fs,args.data_dir, "character_to_prediction_index.json")
landmarks_dir = join_path(fs, args.data_dir, "train_landmarks")

if not fs.exists(train_csv):
    raise FileNotFoundError(f"Missing {train_csv}")
if not fs.exists(vocab_json):
    raise FileNotFoundError(f"Missing {vocab_json}")
if not fs.isdir(landmarks_dir):
    raise FileNotFoundError(f"Missing folder {landmarks_dir}")


In [11]:
 # Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [12]:
# Load vocab mapping (char -> id) in CTC-compatible form
letter_to_int, int_to_letter, blank_id = build_ctc_vocab(vocab_json, fs)

In [13]:
# Load train.csv
df = pd.read_csv(train_csv)
required_cols = {"file_id", "sequence_id", "participant_id", "phrase"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"train.csv is missing columns: {missing}")

df.shape

(67208, 5)

In [14]:
# Filter by parquets you actually downloaded
have_ids = existing_file_ids(landmarks_dir, fs)
if not have_ids:
    raise ValueError(
        f"No parquet files found in {landmarks_dir}. "
        f"Download a few like 0.parquet, 1.parquet, etc."
    )
df = df[df["file_id"].isin(have_ids)].copy()
print(f"Rows after filtering to available parquets ({len(have_ids)} file_ids): {len(df)}")

Rows after filtering to available parquets (68 file_ids): 67208


In [15]:
if args.max_phrase_len > 0:
    df = df[df["phrase"].astype(str).str.len() <= args.max_phrase_len].copy()
    print(f"Rows after filtering by max_phrase_len={args.max_phrase_len}: {len(df)}")
    if len(df) == 0:
        raise ValueError("No rows left after max_phrase_len filtering.")

In [16]:
if args.overfit_subset > 0:
    n_subset = min(args.overfit_subset, len(df))
    overfit_df = df.sample(n=n_subset, random_state=args.seed).copy()
    overfit_df["encoded"] = overfit_df["phrase"].apply(lambda x: encode_phrase(str(x), letter_to_int))
    train_df = overfit_df.copy()
    val_df = overfit_df.copy()
    print(f"Overfit mode enabled: using same {n_subset} samples for train and val")
else:
    # Split by participant_id
    train_df, val_df = split_by_participant(df, val_ratio=args.val_ratio, seed=args.seed)

    # Encode targets
    train_df["encoded"] = train_df["phrase"].apply(lambda x: encode_phrase(str(x), letter_to_int))
    val_df["encoded"] = val_df["phrase"].apply(lambda x: encode_phrase(str(x), letter_to_int))

    # Sample small subsets for local dev
    if args.train_size and args.train_size < len(train_df):
        train_df = train_df.sample(args.train_size, random_state=args.seed)
    if args.val_size and args.val_size < len(val_df):
        val_df = val_df.sample(args.val_size, random_state=args.seed)

print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)}")

Train samples: 200 | Val samples: 200


In [17]:
# Datasets / loaders
train_ds = ASLRightHandDataset(train_df, landmarks_dir=landmarks_dir, max_frames=args.max_frames)
val_ds = ASLRightHandDataset(val_df, landmarks_dir=landmarks_dir, max_frames=args.max_frames)

train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0)

In [18]:
# Model
# Right-hand features are typically 21 landmarks * 3 coords = 63
input_dim = 63
hidden_dim = args.hidden_dim
# output_dim must match max id + 1 (including blank=0)
output_dim = max(int_to_letter.keys()) + 1

model = EmbeddedRNN(input_dim, hidden_dim, output_dim).to(device)

In [19]:
criterion = nn.CTCLoss(blank=blank_id, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

In [20]:
# Tracking setup
run_name = args.run_name or datetime.now().strftime("run_%Y%m%d_%H%M%S")

# TensorBoard
log_path = os.path.join(args.logdir, run_name)
os.makedirs(log_path, exist_ok=True)
writer = SummaryWriter(log_path)
print(f"TensorBoard logdir: {log_path}")

TensorBoard logdir: artifacts/logs/run_20260307_234019


In [21]:
# Weights & Biases
wandb_enabled = args.use_wandb and args.wandb_mode != "disabled"
if args.use_wandb and wandb is None:
    raise ImportError("wandb is not installed. Run: pip install wandb")

if wandb_enabled:
    wandb.init(
        project=args.wandb_project,
        entity=args.wandb_entity,
        name=args.wandb_run_name or run_name,
        config=vars(args),
        mode=args.wandb_mode,
        tags=parse_wandb_tags(args.wandb_tags),
    )

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: julien-cojan (inaki-rodriguez-reyes-upc-universidad-peruana-de-ciencia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [22]:
global_step = 0
for epoch in range(args.epochs):
    model.train()
    losses = []
    blank_ratios = []
    in_tar_ratios = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{args.epochs}", leave=False)

    for batch in pbar:
        if batch is None:
            continue
        X, Y, in_lens, tar_lens = batch
        X = X.to(device)

        optimizer.zero_grad()
        log_probs = model(X)  # (T, B, C)

        loss = criterion(log_probs, Y, in_lens, tar_lens)
        loss.backward()
        optimizer.step()

        # Simple diagnostics: blank-token dominance and input/target length ratio.
        with torch.no_grad():
            pred_ids = torch.argmax(log_probs, dim=2)  # (T, B)
            blank_mask = (pred_ids == blank_id).float()
            blank_ratios.append(float(blank_mask.mean().item()))
            ratio_vals = (in_lens.float() / tar_lens.float().clamp_min(1.0)).detach().cpu()
            in_tar_ratios.append(float(ratio_vals.mean().item()))

        loss_val = float(loss.item())
        losses.append(loss_val)

        writer.add_scalar("loss/train_step", loss_val, global_step)
        if wandb_enabled:
            wandb.log({"loss/train_step": loss_val, "global_step": global_step}, step=global_step)

        global_step += 1
        pbar.set_postfix(loss=loss_val)

    mean_loss = float(sum(losses) / max(1, len(losses)))
    mean_blank_ratio = float(sum(blank_ratios) / max(1, len(blank_ratios)))
    mean_in_tar_ratio = float(sum(in_tar_ratios) / max(1, len(in_tar_ratios)))
    writer.add_scalar("loss/train", mean_loss, epoch)
    writer.add_scalar("diag/blank_ratio_pred", mean_blank_ratio, epoch)
    writer.add_scalar("diag/input_target_len_ratio", mean_in_tar_ratio, epoch)
    print(f"Epoch {epoch + 1}: train loss={mean_loss:.4f}")

    train_metrics = None
    if args.eval_train_metrics:
        train_metrics = evaluate_metrics(
            model,
            train_loader,
            int_to_letter=int_to_letter,
            device=device,
            blank_id=blank_id,
        )
        writer.add_scalar("cer/train", train_metrics["cer"], epoch)
        writer.add_scalar("wer/train", train_metrics["wer"], epoch)
        writer.add_scalar("sequence_accuracy/train", train_metrics["sequence_accuracy"], epoch)
        writer.add_scalar("avg_edit_distance/train", train_metrics["avg_edit_distance"], epoch)

    # Validation metrics
    metrics = evaluate_metrics(
        model,
        val_loader,
        int_to_letter=int_to_letter,
        device=device,
        blank_id=blank_id,
    )
    writer.add_scalar("cer/val", metrics["cer"], epoch)
    writer.add_scalar("wer/val", metrics["wer"], epoch)
    writer.add_scalar("sequence_accuracy/val", metrics["sequence_accuracy"], epoch)
    writer.add_scalar("avg_edit_distance/val", metrics["avg_edit_distance"], epoch)

    if wandb_enabled:
        payload = {
            "epoch": epoch + 1,
            "loss/train": mean_loss,
            "diag/blank_ratio_pred": mean_blank_ratio,
            "diag/input_target_len_ratio": mean_in_tar_ratio,
            "cer/val": metrics["cer"],
            "wer/val": metrics["wer"],
            "sequence_accuracy/val": metrics["sequence_accuracy"],
            "avg_edit_distance/val": metrics["avg_edit_distance"],
            "global_step": global_step,
        }
        if train_metrics is not None:
            payload.update(
                {
                    "cer/train": train_metrics["cer"],
                    "wer/train": train_metrics["wer"],
                    "sequence_accuracy/train": train_metrics["sequence_accuracy"],
                    "avg_edit_distance/train": train_metrics["avg_edit_distance"],
                }
            )
        wandb.log(payload, step=global_step)

    if train_metrics is not None:
        print(
            f"Epoch {epoch + 1}: "
            f"train CER={train_metrics['cer']:.4f} | "
            f"WER={train_metrics['wer']:.4f} | "
            f"ExactMatch={train_metrics['sequence_accuracy']:.4f} | "
            f"AvgEditDist={train_metrics['avg_edit_distance']:.4f}"
        )
    print(
        f"Epoch {epoch + 1}: "
        f"diag blank_ratio_pred={mean_blank_ratio:.4f} | "
        f"input/target ratio={mean_in_tar_ratio:.2f}"
    )

    print(
        f"Epoch {epoch + 1}: "
        f"val CER={metrics['cer']:.4f} | "
        f"WER={metrics['wer']:.4f} | "
        f"ExactMatch={metrics['sequence_accuracy']:.4f} | "
        f"AvgEditDist={metrics['avg_edit_distance']:.4f}"
    )

    # Save checkpoint
    ckpt_path = os.path.join("artifacts", "models", f"{run_name}_epoch{epoch + 1}.pt")
    os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": vars(args),
        },
        ckpt_path,
    )

KeyboardInterrupt: 

In [23]:
if wandb_enabled:
    log_examples_to_wandb(
        model=model,
        dataloader=val_loader,
        int_to_letter=int_to_letter,
        device=device,
        blank_id=blank_id,
        global_step=global_step,
        split_name="val",
        n_examples=5,
    )

writer.close()
if wandb_enabled:
    wandb.finish()

Logging 5 GT/PRED examples (val):
[1] GT: 57 north 115th glen
    PRED: 
[2] GT: 5417 brux rd
    PRED: 
[3] GT: +40-965-37-62-096-65
    PRED: 
[4] GT: 615-423-1108
    PRED: 
[5] GT: +94-481-47-8844-742
    PRED: 


global_step,▁▂▄▅▇█
loss/train_step,▅▁█▂▁
global_step,5
loss/train_step,13.48568
